In [ ]:
import networkx as nx
import numpy as np
from collections import defaultdict
import time
import pandas as pd

def load_graph(path):
    # Detect and handle CSV files
    if path.endswith('.csv'):
        try:
            # Using sep=None and engine='python' forces Pandas to auto-detect
            # the delimiter (handles semicolons, tabs, etc.)
            df = pd.read_csv(path, sep=None, engine='python')

            if len(df.columns) < 2:
                raise ValueError("Not enough columns")

            source_col = df.columns[0]
            target_col = df.columns[1]
            G = nx.from_pandas_edgelist(df, source=source_col, target=target_col)
        except Exception:
            # Fallback for heavily malformed CSVs
            G = nx.read_edgelist(path, delimiter=',', comments='#')
    else:
        # Handle standard whitespace-separated edgelists
        G = nx.read_edgelist(path, comments='#')

    G = nx.convert_node_labels_to_integers(G)
    return G

def generate_attributes(G, dim=5):
    return {node: np.random.rand(dim) for node in G.nodes()}

def get_genome_length(G):
    # Optimised: The sum of all degrees is exactly the genome length needed
    return sum(dict(G.degree()).values())

def initialize_population(genome_length, pop_size):
    return np.random.uniform(-1, 1, (pop_size, genome_length))

def decode_individual(G, node_neighbors, x):
    """
    Uses Locus-Based Adjacency. Instead of direct community IDs,
    nodes pick a neighbor to link to. Components form the communities.
    """
    choices = []
    index = 0

    for node in G.nodes():
        neighbors = node_neighbors[node]
        deg = len(neighbors)

        if deg == 0:
            continue

        xi = x[index : index + deg]
        index += deg

        # Removed softmax/sigmoid: argmax of raw weights is mathematically identical and much faster
        chosen_neighbor = neighbors[np.argmax(xi)]
        choices.append((node, chosen_neighbor))

    return choices

def get_communities(G, choices):
    # The communities are the connected components of the chosen links
    temp_G = nx.Graph()
    temp_G.add_nodes_from(G.nodes())
    temp_G.add_edges_from(choices)
    return list(nx.connected_components(temp_G))

def compute_modularity(G, communities):
    if len(communities) <= 1:
        return -1.0
    return nx.algorithms.community.quality.modularity(G, communities)

def attribute_similarity(communities, attr):
    score = 0
    count = 0

    for comp in communities:
        nodes = list(comp)
        if len(nodes) > 1:
            features = np.array([attr[n] for n in nodes])
            centroid = features.mean(axis=0)
            score += np.sum(np.linalg.norm(features - centroid, axis=1))
            count += len(nodes)

    return score / (count + 1e-6)

def evolve(G, attr, pop_size=10, generations=5):
    genome_length = get_genome_length(G)
    population = initialize_population(genome_length, pop_size)

    # Pre-caching neighbors saves millions of list() calls during decoding
    node_neighbors = {n: list(G.neighbors(n)) for n in G.nodes()}

    best_modularity = -1.0
    best_num_communities = 0

    for gen in range(generations):
        new_pop = []

        for i in range(pop_size):
            x = population[i]

            # Mutation
            y = x + 0.5 * (np.random.rand(len(x)) - 0.5)

            choices = decode_individual(G, node_neighbors, y)
            communities = get_communities(G, choices)
            num_comms = len(communities)

            mod = compute_modularity(G, communities)

            # Fitness: Heavily weight modularity. Scale down attribute similarity
            # so it acts only as a tie-breaker, forcing high modularity.
            f1 = -mod
            f2 = attribute_similarity(communities, attr) * 0.001
            fitness = f1 + f2

            # Append num_comms to track it
            new_pop.append((y, fitness, mod, num_comms))

        # Sort by lowest fitness score (highest modularity)
        new_pop.sort(key=lambda item: item[1])
        population = np.array([item[0] for item in new_pop[:pop_size]])

        # Track the modularity and communities of the fittest individual
        best_modularity = new_pop[0][2]
        best_num_communities = new_pop[0][3]

    return best_modularity, best_num_communities

# --- Execution Block ---
datasets = [
    "/content/Dataset_CyberCrime_Sean.csv",
    "/content/london_crime_by_lsoa.csv",
    "/content/twitter_combined.txt.gz",
    "/content/facebook_combined.txt.gz", # Double-check this filename in your folder
    "/content/Meetings.csv",
    "/content/Phone_Calls.csv",
    "/content/Roles.csv",
    "/content/Sicilian Mafia.csv"
]

# Updated formatting to include Nodes, Edges, and Comms
print(f"{'Dataset':<32} | {'Nodes':<6} | {'Edges':<7} | {'Comms':<6} | {'Modularity':<10} | {'Runtime (s)':<12}")
print("-" * 88)

for ds in datasets:
    try:
        G = load_graph(ds)

        # Extract Node and Edge counts
        num_nodes = G.number_of_nodes()
        num_edges = G.number_of_edges()

        attr = generate_attributes(G)

        start_time = time.time()
        # Note: Increase 'generations' to 50+ in your final run to achieve maximum possible modularity
        final_modularity, num_communities = evolve(G, attr, pop_size=10, generations=5)
        runtime = time.time() - start_time

        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {num_nodes:<6} | {num_edges:<7} | {num_communities:<6} | {final_modularity:<10.4f} | {runtime:<12.2f}")

    # Added specific catch for FileNotFoundError to keep the table clean
    except FileNotFoundError:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'Not Found':<10} | {'N/A':<12}")

    except Exception as e:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'Error':<10} | {str(e)[:12]:<12}")

Dataset                          | Nodes  | Edges   | Comms  | Modularity | Runtime (s) 
----------------------------------------------------------------------------------------
Dataset_CyberCrime_Sean.csv      | 136    | 161     | 17     | 0.6115     | 0.10        
london_crime_by_lsoa.csv         | 4868   | 4835    | 33     | 0.9676     | 3.34        
twitter_combined.txt.gz          | 81306  | 1342310 | 1870   | 0.2546     | 150.01      
facebook_combined.txt.gz         | 4039   | 88234   | 124    | 0.4753     | 8.04        
Meetings.csv                     | 95     | 248     | 12     | 0.4536     | 0.08        
Phone_Calls.csv                  | 92     | 119     | 8      | 0.5986     | 0.06        
Roles.csv                        | 161    | 134     | 27     | 0.7742     | 0.12        
Sicilian Mafia.csv               | 143    | 325     | 17     | 0.4466     | 0.09        
